Chatbot using Hugging Face Transformers
This project implements a conversational chatbot using a pre-trained transformer model from Hugging Face.

**Objective**

To build a chatbot that can interact with users using natural language and generate meaningful responses using transformer-based models.

In [1]:
# Install required libraries (run once)
!pip install transformers torch

In [2]:
# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [8]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.resize_token_embeddings(len(tokenizer))

print("Model loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [9]:
def run_chatbot():

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Store conversation history
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Give short, clear, and accurate answers."
        }
    ]

    while True:
        user_input = input("You: ")

        # Exit condition (allowed)
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! ")
            break

        # Add user input
        chat_history.append({"role": "user", "content": user_input})

        # Convert to model format
        inputs = tokenizer.apply_chat_template(
            chat_history,
            return_tensors="pt",
            add_generation_prompt=True
        )

        # Move input to model device
        input_ids = inputs["input_ids"].to(model.device)

        # Generate response
        outputs = model.generate(
            input_ids,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

        # Decode response
        response = tokenizer.decode(
            outputs[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        )


        response = response.replace("  ", " ")
        response = response.split("\n")[0]
        response = response.split("###")[0]
        response = response.strip()

        if "." in response:
            response = response.split(".")[0] + "."

        # Safety fallback
        if len(response.strip()) < 5:
            response = "I'm here to help! Could you please ask your question again?"

        print("Chatbot:", response)


        chat_history.append({"role": "assistant", "content": response})



run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: hello
Chatbot: Hello! How can I assist you today?
You: Who created Python?
Chatbot: Python was created by Guido van Rossum, who is known for his contributions to the language's design and development.
You:  What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and act like humans.
You: do you know about machine learning?
Chatbot: Yes, Machine Learning is a subfield of artificial intelligence concerned with the development of algorithms that enable computers to learn from data without being explicitly programmed.
You: Thank You
Chatbot: You're welcome! If you have any more questions, feel free to ask.
You: exit
Chatbot: Goodbye! 
